# Lab 4b: Evaluating an Image Segmentation Model

**Before you start:** If you are using Google Colab, remember to enable GPU.

This is **Part 2 of 3** in a mini-series on evaluating deep learning models.

Unlike Lab 4a, **you will not train anything in this notebook**. Instead, you'll take a model
someone else already trained, run it on a new dataset it has never seen, and figure out — using
proper metrics — how good it actually is. This is an extremely common real-world scenario: before
you invest time fine-tuning or building your own model, you first check whether an existing
pretrained model is already good enough (or how far off it is).

## Your task

1. Download and describe the Oxford-IIIT Pet segmentation dataset.
2. Load a pretrained semantic segmentation model and run it on the dataset.
3. Convert the model's output and the ground-truth annotation into comparable binary masks.
4. Evaluate the model using **Intersection over Union (IoU)** and the **Dice score**, and visualize some predictions.


In [ ]:
# Setup
!pip install -q torchmetrics


In [ ]:
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

import torchvision
from torchvision.models.segmentation import deeplabv3_resnet50, DeepLabV3_ResNet50_Weights

from torchmetrics.segmentation import DiceScore
from torchmetrics.classification import BinaryJaccardIndex

import numpy as np
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

torch.manual_seed(0)


## Task 1: The dataset

We switch datasets for this lab: [**Oxford-IIIT Pet**](https://www.robots.ox.ac.uk/~vgg/data/pets/) (Parkhi et al., 2012),
a dataset of ~7,349 photos of cats and dogs across 37 breeds. The dataset ships with pixel-level **segmentation trimaps** for every image — exactly what we need to evaluate a segmentation model.

As in Lab 4a, it's worth describing your dataset before using it:

- What are you trying to predict? (Hint: this is a *semantic segmentation* task (pixel-wise classification) — see Lecture 8.)
- How many images are there, and how are they split into train/test?
- What do the annotations actually encode? (see below)
- Why might this dataset be a reasonable choice for testing a pretrained segmentation model, compared to, say, testing directly on medical images?

### 1.1 Download

Each trimap is a single-channel PNG with **3 possible pixel values**:

| Value | Meaning |
|---|---|
| 1 | Pet (foreground) |
| 2 | Background |
| 3 | Border / undefined (boundary pixels where the annotator wasn't confident) |


In [ ]:
# We only need a handful of images to get meaningful evaluation numbers, so we evaluate on a
# random subset of the test split rather than all ~3,669 images (feel free to raise this if you have GPU time).
subset_size = 200

test_dataset = torchvision.datasets.OxfordIIITPet(
    root="./data",
    split="test",
    target_types=["segmentation", "binary-category"],  # binary-category: 0 = Cat, 1 = Dog
    download=True,
)

rng = np.random.default_rng(0)
subset_indices = rng.choice(len(test_dataset), size=subset_size, replace=False)
print(f"Using {subset_size} of {len(test_dataset)} test images.")
print("Species classes:", test_dataset.bin_classes)


In [ ]:
# Look at a couple of examples: image + trimap side by side.
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for col, idx in enumerate(subset_indices[:4]):
    image, (trimap, species) = test_dataset[idx]
    trimap = np.array(trimap)

    axes[0, col].imshow(image)
    axes[0, col].set_title(test_dataset.bin_classes[species])
    axes[0, col].axis("off")

    axes[1, col].imshow(trimap, cmap="viridis", vmin=1, vmax=3)
    axes[1, col].set_title("trimap")
    axes[1, col].axis("off")

print("Unique trimap values in this image:", np.unique(trimap))
plt.tight_layout()
plt.show()


## Task 2: Load a pretrained segmentation model

We'll use **DeepLabV3** with a ResNet-50 backbone, pretrained by `torchvision` on a 21-class subset
of COCO that matches the Pascal VOC classes. Conveniently, two of those 21 classes are exactly
what we need:

```python
VOC_CLASSES = ['__background__', 'aeroplane', 'bicycle', 'bird', 'boat', 'bottle', 'bus', 'car',
               'cat', 'chair', 'cow', 'diningtable', 'dog', 'horse', 'motorbike', 'person',
               'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor']
# index 8  -> cat
# index 12 -> dog
```

This model has never seen the Oxford-IIIT Pet dataset during training — we're evaluating it
**zero-shot**, i.e. purely off-the-shelf.


In [ ]:
CAT_IDX, DOG_IDX = 8, 12

weights = ???  # hint: DeepLabV3_ResNet50_Weights.DEFAULT
model = deeplabv3_resnet50(weights=weights)
model.eval().to(device)

preprocess = weights.transforms()  # applies the exact resize/normalization this model was trained with
print(preprocess)


## Task 3: From model output to a comparable mask

The model outputs a score (logit) for **all 21 classes at every pixel**. To evaluate it on our
binary (pet vs. background) problem, we need to:

1. Run the image through the model and apply `softmax` over the class dimension to get per-pixel probabilities.
2. Pick out the probability channel that matches the image's species (index 8 for cats, index 12 for dogs) — we know the species from the dataset's `binary-category` label, we're not asking the model to guess it.
3. Threshold that probability map (e.g. at 0.5) to get a binary predicted mask.
4. Convert the ground-truth trimap into a binary mask. We'll treat trimap value 2 (background) as background, and both 1 (pet) and 3 (border) as foreground — the border pixels are still part of the pet, just with fuzzy edges. (Hint: PyTorch supports expressions like `(trimap == 1) | (trimap == 3)`).

**Tip:** the model's output resolution may not exactly match the input image resolution — check the shapes, and use [`F.interpolate`](https://docs.pytorch.org/docs/2.14/generated/torch.nn.functional.interpolate.html) to resize if needed.


In [ ]:
@torch.no_grad()
def predict_mask(image_pil, species_idx):
    '''Returns a (H, W) boolean predicted foreground mask for one PIL image.'''
    input_tensor = preprocess(image_pil).unsqueeze(0).to(device)
    output = model(input_tensor)["out"]  # shape (1, 21, h, w) -- h, w may differ from the original image!

    output = ???  # resize `output` to (image_pil.height, image_pil.width) if its spatial size differs -- see F.interpolate
    probs = torch.softmax(output, dim=1)
    species_prob = probs[0, species_idx]  # (H, W) probability that each pixel belongs to this image's species

    predicted_mask = species_prob > ???  # threshold, e.g. 0.5
    return predicted_mask.cpu().numpy()


def trimap_to_mask(trimap):
    '''Converts a raw trimap (values 1/2/3) into a binary foreground mask.'''
    trimap = np.array(trimap)
    return ???  # True where trimap is 1 (pet) or 3 (border), False where it's 2 (background)


In [ ]:
# Quick visual sanity check on one example before running the full evaluation loop.
image, (trimap, species) = test_dataset[subset_indices[0]]

# `species` from the dataset is 0=Cat/1=Dog -- map it to the segmentation model's class indices:
species_idx = CAT_IDX if species == 0 else DOG_IDX
pred_mask = predict_mask(image, species_idx)
gt_mask = trimap_to_mask(trimap)

fig, axes = plt.subplots(1, 3, figsize=(10, 4))
axes[0].imshow(image); axes[0].set_title("image"); axes[0].axis("off")
axes[1].imshow(gt_mask, cmap="gray"); axes[1].set_title("ground truth"); axes[1].axis("off")
axes[2].imshow(pred_mask, cmap="gray"); axes[2].set_title("prediction"); axes[2].axis("off")
plt.tight_layout()
plt.show()


## Task 4: Metrics — IoU and Dice score

Two standard metrics for comparing a predicted mask $P$ to a ground-truth mask $G$:

$$\text{IoU} = \frac{|P \cap G|}{|P \cup G|} \qquad\qquad \text{Dice} = \frac{2|P \cap G|}{|P| + |G|}$$

- **IoU** (Intersection over Union, a.k.a. the Jaccard index) is the metric mentioned in Lecture 8 for localization, detection, *and* segmentation.
- **Dice** is closely related — in fact for binary masks, $\text{Dice} = \dfrac{2 \cdot \text{IoU}}{1 + \text{IoU}}$ — and is extremely common in medical image segmentation specifically (see the "Medical Image Segmentation" project idea).

Both range from 0 (no overlap) to 1 (perfect overlap). We'll use `torchmetrics` to compute both, accumulated over our whole evaluation subset.

**Documentation:** [`BinaryJaccardIndex`](https://lightning.ai/docs/torchmetrics/stable/classification/jaccard_index) | [`DiceScore`](https://lightning.ai/docs/torchmetrics/stable/segmentation/dice)


In [ ]:
from torchmetrics.functional.segmentation import dice_score as dice_score_fn

iou_metric = BinaryJaccardIndex()
dice_metric = DiceScore(num_classes=2, average="none", input_format="index")  # index 0: background, index 1: foreground

per_image_dice = []  # keep per-image scores too, so we can look at best/worst cases later

for idx in subset_indices:
    image, (trimap, species) = test_dataset[idx]
    species_idx = CAT_IDX if species == 0 else DOG_IDX

    pred_mask = predict_mask(image, species_idx)
    gt_mask = trimap_to_mask(trimap)

    pred_tensor = torch.from_numpy(pred_mask.astype(int)).unsqueeze(0)  # add a batch dim: (1, H, W)
    gt_tensor = torch.from_numpy(gt_mask.astype(int)).unsqueeze(0)

    iou_metric.update(pred_tensor, gt_tensor)
    dice_metric.update(pred_tensor, gt_tensor)
    per_image_dice.append(dice_score_fn(pred_tensor, gt_tensor, num_classes=2, average="none", input_format="index")[0, 1].item())

print(f"Mean IoU (foreground):  {iou_metric.compute().item():.4f}")
print(f"Mean Dice (foreground): {dice_metric.compute()[1].item():.4f}")


In [ ]:
# Visualize the best and worst predictions by Dice score.
order = np.argsort(per_image_dice)
worst_idx = [subset_indices[i] for i in order[:3]]
best_idx = [subset_indices[i] for i in order[-3:]]

fig, axes = plt.subplots(2, 3, figsize=(10, 7))
for row, (title, indices) in enumerate([("Worst", worst_idx), ("Best", best_idx)]):
    for col, idx in enumerate(indices):
        image, (trimap, species) = test_dataset[idx]
        species_idx = CAT_IDX if species == 0 else DOG_IDX
        pred_mask = predict_mask(image, species_idx)

        axes[row, col].imshow(image)
        axes[row, col].imshow(pred_mask, cmap="Reds", alpha=0.4)
        axes[row, col].set_title(f"{title} #{col+1}")
        axes[row, col].axis("off")
plt.tight_layout()
plt.show()


## Discussion

- Are IoU and Dice highly correlated for your results? (They should be, given the formula relating them — but it's worth checking with your own numbers.)
- Look at your worst-scoring images. Do you see a pattern (e.g. multiple animals in frame, unusual pose, cluttered background, the animal only partially visible)?
- Would you expect **fine-tuning** this model on Oxford-IIIT Pet training data to help a lot, a little, or not at all? Why?
- **Optional further exploration:** `torchvision` also ships pretrained *instance* segmentation models (e.g. Mask R-CNN). How would you need to change the evaluation approach above if you used one of those instead of DeepLabV3 (which does *semantic* segmentation)?

---

**Next up:** `Lab 4c` covers object detection, evaluated with IoU-based box matching and mean Average Precision (mAP).